### NOTES

You need to have the smol dataset on your google drive in MyDrive/smol.

In [133]:
# CONFIG
LANGUAGES = ["es", "lij", "mfe",
             "is", "pcm", "kri",
             "bm", "dyu", "sus",
             "ach", "alz", "luo",
             "sw", "tn", "bem",
             ]

USE_GATITOS = False # May impact accuracy.
USE_SMOLDOCS = True # May give better results for low-resource languages mBART-50 doesn't know, but not all languages have docs, check documentation.
SEED = 133 # The seed used to pick the training/test/val split out of all the data.

EPOCHS = 1 # Currently this gives the best results for the time invested.
TRAINING_BATCH_SIZE = 8 # Saves memory.
LIMIT_TEST_SIZES = True # Limits the size of test and val split to 173 / 86.

TYPOLOGY_AWARE = False # Whether we use the base model or our typology aware model.
USE_SCRIPT = True
USE_FAMILY = True
USE_REGION = True

##Typology Aware mBART


In [134]:
# COLLAB
from google.colab import drive
drive.mount('/content/drive')

# INSTALLS
!pip install evaluate
!pip install sacrebleu

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [135]:
# IMPORTS
!unzip -o "/content/drive/MyDrive/smol.zip" -d "/content/"
smol_path = "/content/smol/"

import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset, concatenate_datasets
from transformers import MBartForConditionalGeneration, MBart50Tokenizer
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import DataCollatorForSeq2Seq
import evaluate
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
import os
import random


Archive:  /content/drive/MyDrive/smol.zip
  inflating: /content/smol/.git/config  
  inflating: /content/smol/.git/description  
  inflating: /content/smol/.git/FETCH_HEAD  
  inflating: /content/smol/.git/HEAD  
  inflating: /content/smol/.git/hooks/applypatch-msg.sample  
  inflating: /content/smol/.git/hooks/commit-msg.sample  
  inflating: /content/smol/.git/hooks/fsmonitor-watchman.sample  
  inflating: /content/smol/.git/hooks/post-update.sample  
  inflating: /content/smol/.git/hooks/pre-applypatch.sample  
  inflating: /content/smol/.git/hooks/pre-commit.sample  
  inflating: /content/smol/.git/hooks/pre-merge-commit.sample  
  inflating: /content/smol/.git/hooks/pre-push.sample  
  inflating: /content/smol/.git/hooks/pre-rebase.sample  
  inflating: /content/smol/.git/hooks/pre-receive.sample  
  inflating: /content/smol/.git/hooks/prepare-commit-msg.sample  
  inflating: /content/smol/.git/hooks/push-to-checkout.sample  
  inflating: /content/smol/.git/hooks/sendemail-validat

In [136]:
class TypologyAwareMBart(nn.Module):
    def __init__(self, base_model, script_classes, family_classes, region_classes):
        super().__init__()
        self.model = base_model
        self.hidden = base_model.config.d_model

        feature_dim = self.hidden // 4

        # SCRIPT
        self.script_emb = nn.Embedding(script_classes, feature_dim)
        if not USE_SCRIPT:
          print("Ignoring script embeddings.")
          with torch.no_grad():
              self.script_emb.weight.zero_()
              self.script_emb.weight.requires_grad = False

        # FAMILY
        self.family_emb = nn.Embedding(family_classes, feature_dim)
        if not USE_FAMILY:
          print("Ignoring family embeddings.")
          with torch.no_grad():
              self.family_emb.weight.zero_()
              self.family_emb.weight.requires_grad = False

        # REGION
        self.region_emb = nn.Embedding(region_classes, feature_dim)
        if not USE_REGION:
          print("Ignoring region embeddings.")
          with torch.no_grad():
              self.region_emb.weight.zero_()
              self.region_emb.weight.requires_grad = False

        self.proj = nn.Linear(3 * feature_dim, self.hidden)

    def forward(self, input_ids, attention_mask, labels,
                script_ids, family_ids, region_ids):

        batch_size, seq_len = input_ids.size()

        # typology embeddings
        s = self.script_emb(script_ids).squeeze(1)
        f = self.family_emb(family_ids).squeeze(1)
        r = self.region_emb(region_ids).squeeze(1)

        typology_vec = torch.cat([s, f, r], dim=-1)
        typology_vec = self.proj(typology_vec)
        typology_tokens = typology_vec.unsqueeze(1).expand(batch_size, seq_len, self.hidden)

        inputs_embeds = self.model.model.encoder.embed_tokens(input_ids)
        inputs_embeds = inputs_embeds + typology_tokens

        return self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels
        )

    def generate(self, input_ids, attention_mask, script_ids, family_ids,
                 region_ids, forced_bos_token_id=None, max_length=128, **kwargs):
        batch_size, seq_len = input_ids.size()

        s = self.script_emb(script_ids).squeeze(1)
        f = self.family_emb(family_ids).squeeze(1)
        r = self.region_emb(region_ids).squeeze(1)

        typology_vec = torch.cat([s, f, r], dim=-1)
        typology_vec = self.proj(typology_vec)
        typology_tokens = typology_vec.unsqueeze(1).expand(
            batch_size, seq_len, self.hidden
        )

        inputs_embeds = (
            self.model.model.encoder.embed_tokens(input_ids)
            + typology_tokens
        )

        return self.model.generate(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            forced_bos_token_id=forced_bos_token_id,
            max_length=max_length,
            **kwargs
        )

In [137]:
SCRIPT_MAP = {"Latin": 0, "Devanagari": 1}
FAMILY_MAP = {
    "Germanic": 0, "Italic": 1, "Central-Mande": 2,
    "Indo-Aryan": 3, "Lwoo": 4, "East-Bantu": 5,
}
REGION_MAP = {"Europe": 0, "Africa": 1, "SouthAsia": 2}

LANG_META = {
    "en": {"script": "Latin", "family": "Germanic", "region": "Europe"},
    "es": {"script": "Latin", "family": "Italic", "region": "Europe"},
    "lij": {"script": "Latin", "family": "Italic", "region": "Europe"},
    "mfe": {"script": "Latin", "family": "Italic", "region": "Africa"},
    "is": {"script": "Latin", "family": "Germanic", "region": "Europe"},
    "pcm": {"script": "Latin", "family": "Germanic", "region": "Africa"},
    "kri": {"script": "Latin", "family": "Germanic", "region": "Africa"},
    "bm": {"script": "Latin", "family": "Central-Mande", "region": "Africa"},
    "dyu": {"script": "Latin", "family": "Central-Mande", "region": "Africa"},
    "sus": {"script": "Latin", "family": "Central-Mande", "region": "Africa"},
    "ach": {"script": "Latin", "family": "Lwoo", "region": "Africa"},
    "alz": {"script": "Latin", "family": "Lwoo", "region": "Africa"},
    "luo": {"script": "Latin", "family": "Lwoo", "region": "Africa"},
    "sw": {"script": "Latin", "family": "East-Bantu", "region": "Africa"},
    "tn": {"script": "Latin", "family": "East-Bantu", "region": "Africa"},
    "bem": {"script": "Latin", "family": "East-Bantu", "region": "Africa"},
}


In [138]:
# ===================== HELPER FUNCTIONS =====================================#

# To figure out what languages have smoldocs
def try_load_jsonl(path):
    if os.path.exists(path):
        return load_dataset("json", data_files=path)["train"]
    return None

# Load all the datasets
def load_language_datasets(lang):
    smolsent = try_load_jsonl(f"{smol_path}/smolsent/en_{lang}.jsonl")
    gatitos = None
    smoldoc = None

    if USE_GATITOS:
      gatitos = try_load_jsonl(f"{smol_path}/gatitos/en_{lang}.jsonl")
    if USE_SMOLDOCS:
      smoldoc = try_load_jsonl(f"{smol_path}/smoldoc/en_{lang}.jsonl")

    print(f"Smolsent: {len(smolsent)}")

    # Normalize gatitos ("trgs" -> "trg")
    if gatitos is not None:
        print(f"Gatitos: {len(gatitos)}")
        gatitos = gatitos.map(unify_gatitos)

    # Flatten smoldoc to smolsent
    if smoldoc is not None:
        smoldoc_flat = smoldoc.map(flatten_smoldoc_to_smolsent, batched=True, remove_columns=smoldoc.column_names)
        print(f"Smoldoc: {len(smoldoc_flat)}")
        smolsent = concatenate_datasets([smolsent, smoldoc_flat])

    print("-" * 50)

    return gatitos, smolsent

# Convert 'trgs' list -> single 'trg' string for gatitos
def unify_gatitos(example):
    example["trg"] = example["trgs"][0]
    return example

# Convert smoldoc 'srcs' and 'trgs' into 'src' and 'trg'
def flatten_smoldoc_to_smolsent(batch):
    new_src = []
    new_trg = []
    new_tl  = []

    for srcs, trgs, tl in zip(batch["srcs"], batch["trgs"], batch["tl"]):

        # ensure srcs/trgs match
        if len(srcs) != len(trgs):
            print("MISMATCH BETWEEN TRGS AND SRC")

        for s, t in zip(srcs, trgs):
            new_src.append(s)
            new_trg.append(t)
            new_tl.append(tl)

    return {
        "src": new_src,
        "trg": new_trg,
        "tl":  new_tl
    }

# Create the train, test, and val split
def split_smolsent(ds):
    train_test = ds.train_test_split(test_size=0.30, seed=42)
    train_ds = train_test["train"]
    test_val_ds = train_test["test"]

    val_test = test_val_ds.train_test_split(test_size=2/3, seed=42)
    val_ds = val_test["train"]
    test_ds = val_test["test"]

    return train_ds, val_ds, test_ds

# This way we ensure that we get the same number of test/val entries per language.
def split_smolsent_maxSize(ds, maxTestSize=173, maxValSize=86):
  seed = SEED
  # Ensure reproducibility
  random.seed(seed)

  # Total indices
  all_indices = list(range(len(ds)))
  random.shuffle(all_indices)

  # Slice indices
  test_indices = all_indices[:maxTestSize]
  val_indices  = all_indices[maxTestSize:maxTestSize + maxValSize]
  train_indices = all_indices[maxTestSize + maxValSize:]

  # Create subsets
  test_ds  = ds.select(test_indices)
  val_ds   = ds.select(val_indices)
  train_ds = ds.select(train_indices)

  return train_ds, val_ds, test_ds

# Create the special language token corresponding to a language
def create_language_ID(lang):
    return f"{lang}_XX"

In [139]:
#========================== MAIN FUNCTION LOOP ===============================#

# Temporary structures
training_sets = []
gatitos_sets = []
val_sets = []

# The language key mapped to test set
test_dataset = {}

for lang in LANGUAGES:
  print(f"Processing language: {lang}")

  gatitos, smolsent = load_language_datasets(lang)

  # Remove other columns
  columns_to_keep = ["src", "trg", "tl"]

  gatitosLength = 0
  if gatitos is not None:
      gatitos = gatitos.select_columns(columns_to_keep)
      gatitos_sets.append(gatitos)
      gatitosLength = len(gatitos)

  if smolsent is not None:
      smolsent = smolsent.select_columns(columns_to_keep)

  train, val, test = split_smolsent(smolsent)
  if LIMIT_TEST_SIZES:
    train, val, test = split_smolsent_maxSize(smolsent)

  print(f"Total training entries: {len(train) + gatitosLength}")
  print(f"Total test entries: {len(test)}")
  print(f"Total validation entries: {len(val)}")
  print("-" * 50)

  training_sets.append(train)
  val_sets.append(val)

  test_dataset[lang] = test

# Merge sets into final
train_dataset = concatenate_datasets(training_sets + gatitos_sets)
val_dataset = concatenate_datasets(val_sets)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test sizes:", {k: len(v) for k, v in test_dataset.items()})


Processing language: es
Smolsent: 863
--------------------------------------------------
Total training entries: 604
Total test entries: 173
Total validation entries: 86
--------------------------------------------------
Processing language: lij
Smolsent: 863
Smoldoc: 825
--------------------------------------------------
Total training entries: 1429
Total test entries: 173
Total validation entries: 86
--------------------------------------------------
Processing language: mfe
Smolsent: 863
Smoldoc: 825
--------------------------------------------------
Total training entries: 1429
Total test entries: 173
Total validation entries: 86
--------------------------------------------------
Processing language: is
Smolsent: 863
--------------------------------------------------
Total training entries: 604
Total test entries: 173
Total validation entries: 86
--------------------------------------------------
Processing language: pcm
Smolsent: 863
Smoldoc: 1609
---------------------------------

In [140]:
# Load model
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
base_model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

# Setup language tokens that don't exist in mBART
special_tokens = [f"<{lang}_XX>" for lang in LANGUAGES]
tokenizer.add_tokens(special_tokens, special_tokens=True)
base_model.resize_token_embeddings(len(tokenizer))

# Map language "es" to token ID "es_XX"
LANG_MAP = {lang: create_language_ID(lang) for lang in LANGUAGES}
print(LANG_MAP)

model = base_model

if TYPOLOGY_AWARE:
  print("Building Typology-Aware mBART...")
  model = TypologyAwareMBart(
    base_model,
    script_classes=len(SCRIPT_MAP),
    family_classes=len(FAMILY_MAP),
    region_classes=len(REGION_MAP)
  )

# Detect GPU or use CPU
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model.to(device)

{'es': 'es_XX', 'lij': 'lij_XX', 'mfe': 'mfe_XX', 'is': 'is_XX', 'pcm': 'pcm_XX', 'kri': 'kri_XX', 'bm': 'bm_XX', 'dyu': 'dyu_XX', 'sus': 'sus_XX', 'ach': 'ach_XX', 'alz': 'alz_XX', 'luo': 'luo_XX', 'sw': 'sw_XX', 'tn': 'tn_XX', 'bem': 'bem_XX'}
Using device: cuda


MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250069, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250069, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [141]:
def get_batch_metadata(batch):
  script_ids = []
  family_ids = []
  region_ids = []
  for tl in batch["tl"]:
    # Typology metadata
    meta = LANG_META[tl]
    script_ids.append(SCRIPT_MAP[meta["script"]])
    family_ids.append(FAMILY_MAP[meta["family"]])
    region_ids.append(REGION_MAP[meta["region"]])

  return script_ids, family_ids, region_ids


In [142]:
def evaluate_translation_model(model, tokenizer, test_datasets, batch_size=16):
    """
    Evaluate mBART on multiple languages.

    Args:
        model: HuggingFace mBART model
        tokenizer: HuggingFace tokenizer
        test_datasets: dict of {lang_code: Dataset}
        batch_size: batch size for DataLoader

    Returns:
        results: dict of {lang_code: BLEU score}
    """
    bleu_metric = evaluate.load("sacrebleu")
    chrf_metric = evaluate.load("chrf")

    model.eval()
    device = None
    if TYPOLOGY_AWARE:
      device = next(model.parameters()).device
    else:
      device = model.device
    results = {}

    for lang, dataset in test_datasets.items():
        print(f"Evaluating language: {lang}")
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

        preds = []
        refs  = []

        for batch in tqdm(loader):
            src_texts = batch["src"]
            tgt_texts = batch["trg"]

            # tokenize source
            inputs = tokenizer(src_texts, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)

            # Determine forced_bos_token_id
            tgt_lang = LANG_MAP[lang]
            if tgt_lang in tokenizer.lang_code_to_id:
                bos_id = tokenizer.lang_code_to_id[tgt_lang]
            else:
                bos_id = tokenizer.convert_tokens_to_ids(f"<{tgt_lang}>")

            generated_ids = []
            if TYPOLOGY_AWARE:
              s, f, r = get_batch_metadata(batch)
              script_ids = torch.tensor(s).to(device)
              family_ids = torch.tensor(f).to(device)
              region_ids = torch.tensor(r).to(device)
              # generate translations
              with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    script_ids=script_ids,
                    family_ids=family_ids,
                    region_ids=region_ids,
                    forced_bos_token_id=bos_id,
                    max_length=128
                )

            else:
              # generate translations
              with torch.no_grad():
                  generated_ids = model.generate(
                      **inputs,
                      forced_bos_token_id=bos_id,
                      max_length=128
                  )

            # decode predictions
            decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            decoded_preds = [p.strip() for p in decoded_preds]

            preds.extend(decoded_preds)
            refs.extend([[t.strip()] for t in tgt_texts])

        # compute BLEU
        bleu = bleu_metric.compute(predictions=preds, references=refs)
        chrf = chrf_metric.compute(predictions=preds, references=refs)

        print(f"{lang} BLEU: {bleu['score']:.2f}, chrF: {chrf['score']:.2f}")
        results[lang] = {"bleu": bleu["score"], "chrf": chrf["score"]}

    return results

# Evaluate all languages
#bleu_scores = evaluate_translation_model(model, tokenizer, test_dataset)

In [143]:
# TRAINING

def preprocess_function(batch):

    # Tokenize source (English)
    model_inputs = tokenizer(
        batch["src"],
        max_length=128,
        truncation=True
    )

    # Map tl -> BOS
    bos_ids = []
    if TYPOLOGY_AWARE:
      script_ids = []
      family_ids = []
      region_ids = []

    for tl in batch["tl"]:
        lang = LANG_MAP[tl]
        bos_token = f"<{lang}>"
        bos_ids.append(tokenizer.convert_tokens_to_ids(bos_token))

        meta = LANG_META[tl]
        if TYPOLOGY_AWARE:
          script_ids.append(SCRIPT_MAP[meta["script"]])
          family_ids.append(FAMILY_MAP[meta["family"]])
          region_ids.append(REGION_MAP[meta["region"]])

    # Tokenize target normally (We can't tokenize like the input because mBART might not know language tokenizer... Also we don't need <s> tokens and so on.
    target_ids = [tokenizer.encode(t, truncation=True, max_length=126, add_special_tokens=False) for t in batch["trg"]]

    # Prepend BOS
    labels = []
    for bos, seq in zip(bos_ids, target_ids):
        new_seq = [bos] + seq + [tokenizer.eos_token_id]
        labels.append(new_seq)

    model_inputs["labels"] = labels

    # Inject metadata into batch
    if TYPOLOGY_AWARE:
      model_inputs["script_ids"] = script_ids
      model_inputs["family_ids"] = family_ids
      model_inputs["region_ids"] = region_ids

    return model_inputs

# tokenize everything
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val   = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Args
training_args = Seq2SeqTrainingArguments(
    output_dir="finetuned-mBart-ES",

    per_device_train_batch_size=TRAINING_BATCH_SIZE,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,

    learning_rate=3e-5,

    #label_smoothing_factor=0.1,
    save_safetensors=False,
    save_strategy="epoch",
    predict_with_generate=True,
    logging_strategy="steps",
    logging_steps=20,

    warmup_ratio=0.1, # May be removed?

    report_to="none"
)

# Define the trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# Train!
trainer.train()


Map:   0%|          | 0/32114 [00:00<?, ? examples/s]

Map:   0%|          | 0/1290 [00:00<?, ? examples/s]

/tmp/ipython-input-3453236763.py:79: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
20,7.977000
40,7.634800
60,7.355500
80,7.043100
100,6.633700
120,6.747100
140,6.238100
160,6.359900
180,6.178500
200,6.186000


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=4015, training_loss=3.9553364963935294, metrics={'train_runtime': 593.0811, 'train_samples_per_second': 54.148, 'train_steps_per_second': 6.77, 'total_flos': 2347632207298560.0, 'train_loss': 3.9553364963935294, 'epoch': 1.0})

In [144]:
# Test the model again
bleu_scores = evaluate_translation_model(model, tokenizer, test_dataset)

Evaluating language: es


100%|██████████| 11/11 [00:08<00:00,  1.26it/s]


es BLEU: 29.26, chrF: 56.66
Evaluating language: lij


100%|██████████| 11/11 [00:15<00:00,  1.38s/it]


lij BLEU: 10.86, chrF: 36.18
Evaluating language: mfe


...

sw BLEU: 20.14, chrF: 45.89
Evaluating language: tn


100%|██████████| 11/11 [00:25<00:00,  2.34s/it]


tn BLEU: 1.76, chrF: 15.48
Evaluating language: bem


100%|██████████| 11/11 [00:17<00:00,  1.55s/it]

bem BLEU: 2.14, chrF: 18.09
